# 04. Split outputs by cohort (psychosis-risk vs non-psychosis)

Takes the combined outputs of notebook 03 and writes per-cohort files:

```
output/pca/
  X_features_pca.parquet          (combined)
  X_mask.parquet                  (combined)
  Y_phecode.parquet               (combined)
  psychosis_risk/
    X_features_pca.parquet
    X_mask.parquet
    Y_phecode.parquet
  non_psychosis/
    X_features_pca.parquet
    X_mask.parquet
    Y_phecode.parquet
```

The PCA basis is shared across both cohorts, so the two `X_features_pca` slices
live in the same embedding space and are directly comparable.


In [ ]:
from pathlib import Path
import pandas as pd

PCA_DIR = Path('output/pca')
ROW_ID = 'row_id'
COHORT_GROUP_COL = 'cohort_group'

# Load combined outputs from notebook 03
X = pd.read_parquet(PCA_DIR / 'X_features_pca.parquet')
mask = pd.read_parquet(PCA_DIR / 'X_mask.parquet')
Y = pd.read_parquet(PCA_DIR / 'Y_phecode.parquet')

print('X:', X.shape)
print('mask:', mask.shape)
print('Y:', Y.shape)
print()
print('Cohort distribution in X:')
print(X[COHORT_GROUP_COL].value_counts())


## Split and save

In [ ]:
COHORT_DIRS = {
    'psychosis_risk': 'psychosis_risk',
    'non_psychosis': 'non_psychosis',
}

def safe_dir_name(label):
    return COHORT_DIRS.get(str(label), str(label).lower().replace(' ', '_').replace('-', '_'))

for group_label in X[COHORT_GROUP_COL].dropna().unique().tolist():
    sub_dir = safe_dir_name(group_label)
    out = PCA_DIR / sub_dir
    out.mkdir(exist_ok=True)

    sub_X = X[X[COHORT_GROUP_COL] == group_label].reset_index(drop=True)
    sub_mask = mask[mask[COHORT_GROUP_COL] == group_label].reset_index(drop=True)
    sub_Y = Y[Y[COHORT_GROUP_COL] == group_label].reset_index(drop=True)

    # Sanity: row_ids align across the three tables.
    assert sub_X[ROW_ID].equals(sub_mask[ROW_ID]), f'{group_label}: X/mask row_id mismatch'
    assert sub_X[ROW_ID].equals(sub_Y[ROW_ID]), f'{group_label}: X/Y row_id mismatch'

    sub_X.to_parquet(out / 'X_features_pca.parquet', index=False)
    sub_mask.to_parquet(out / 'X_mask.parquet', index=False)
    sub_Y.to_parquet(out / 'Y_phecode.parquet', index=False)

    print(f'[{group_label}] -> {out}/')
    print(f'   X:    {sub_X.shape}')
    print(f'   mask: {sub_mask.shape}')
    print(f'   Y:    {sub_Y.shape}')
    print()


## Final manifest

In [ ]:
print('All output files:')
for path in sorted(PCA_DIR.rglob('*')):
    if path.is_file():
        size_mb = path.stat().st_size / 1024**2
        print(f'  {str(path):60s}  {size_mb:8.2f} MB')
